# v0.31.0 — Server-side events and stored functions

## 1. Intro

Since v0.31.0 the ORM defines two kinds of objects that SurrealDB runs itself, from Python:

- an **event** (`DEFINE EVENT`) runs SurrealQL inside the database whenever a record of its table is created, updated
  or deleted, whoever writes it — the ORM, a raw query or another client: an audit row, a guard that refuses a write,
  a message for an outbox. Events are declared next to the model, in `SurrealConfigDict(events=[Event(...)])`, and
  applied with `await Model.define_events()`; `SurrealDBConnectionManager.define_event(event, table)` applies one to any
  table;
- a **stored function** (`DEFINE FUNCTION`, `fn::…`) is defined with `SurrealDBConnectionManager.define_function()` and
  called with `call_function()`.

The ORM renders each statement (`Event.ddl()`, `Model.event_ddl()`, `function_ddl()` touch no database), writes names
quoted, checks the expressions for mistakes and applies the statements idempotently (`OVERWRITE`), with `tx=` for a
transaction. The same code runs on SurrealDB 2.x and 3.x; asynchronous events (`ASYNC`, §8) and `$input` are 3.x only.
The notebook runs top to bottom on both lines: only the server line (§2), the texts read back (§7) and the asynchronous
section (§8) differ.

## 2. Connect

`SURREALDB_HOST` and `SURREALDB_PORT` choose the server (default `localhost:8000`), over WebSocket. The events and
functions live in their own database, `examples_events`, reached through the named connection `events_demo`. The
notebook starts by resetting that database, so it can be re-executed in place: `REMOVE` and `DEFINE` go in **one**
query, because on SurrealDB 3.x a bare `REMOVE DATABASE` leaves the session failing until the database is defined
again.

`apply_ddl()` runs SurrealQL through the SDK client's `query_raw()` and checks the status of **every** statement
(`client.query()` checks only the first one's).

In [1]:
import asyncio
import os

from surreal_orm_lite import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")
SERVER_URL = f"ws://{HOST}:{PORT}/rpc"

SurrealDBConnectionManager.set_connection(  # the default connection
    url=SERVER_URL,
    user="root",
    password="root",
    namespace="examples",
    database="examples",
)
SurrealDBConnectionManager.add_connection(  # the database this notebook defines events and functions in
    "events_demo",
    url=SERVER_URL,
    user="root",
    password="root",
    namespace="examples",
    database="examples_events",
)


async def apply_ddl(client, ddl: str) -> int:
    """Run SurrealQL statements and check every one of them; return how many ran."""
    response = await client.query_raw(ddl, {})
    if "error" in response:
        raise RuntimeError(response["error"])
    refused = [result["result"] for result in response["result"] if result["status"] != "OK"]
    if refused:
        raise RuntimeError(refused)
    return len(response["result"])


demo = await SurrealDBConnectionManager.get_client("events_demo")
await apply_ddl(demo, "REMOVE DATABASE IF EXISTS examples_events; DEFINE DATABASE examples_events;")

major = int((await demo.version()).rsplit("-", 1)[-1].split(".")[0])  # "surrealdb-3.3.4" -> 3
print("Connections:", SurrealDBConnectionManager.list_connections())
print(f"Server: SurrealDB {major}.x")

Connections: ['default', 'events_demo']
Server: SurrealDB 3.x


## 3. Events on a model

`Account` declares two events in `SurrealConfigDict(events=[…])`, as the README's first example does:

- `email_audit` writes an `audit_log` row when an update changes the e-mail. Inside an event, `$event` is `'CREATE'`,
  `'UPDATE'` or `'DELETE'`, `$before` and `$after` are the record before and after the write, `$value` is the record;
- `no_negative_balance` is a guard: its `THROW` refuses the write. It names `$event != 'DELETE'`: after a delete
  `$after.balance` is `NONE`, and `NONE < 0` is true, so without it the guard would refuse every delete.

`then` is the SurrealQL the event runs; the ORM always writes it as a block (`{ … }`), so several statements, `LET` and
`THROW` work on both lines. `when` is one expression. Both are written into the statement as they are, since SurrealDB
takes no bound parameter in a definition: they are trusted application code, never user input. The ORM checks them for
mistakes (unbalanced brackets, a comment marker, a second statement where one expression is expected) — not as an
injection defence. `AuditLog` reads the rows the event writes; it declares no field (`extra="allow"`), so a row's
columns are in `model_extra`. Both models run on the `events_demo` connection.

In [2]:
from surreal_orm_lite import BaseSurrealModel, Event, SurrealConfigDict


class Account(BaseSurrealModel):
    model_config = SurrealConfigDict(
        table_name="account",
        connection="events_demo",
        events=[
            Event(
                "email_audit",
                when="$event = 'UPDATE' AND $before.email != $after.email",
                then="CREATE audit_log SET account = $value.id, old = $before.email, new = $after.email",
                comment="Keeps every e-mail change",
            ),
            Event(
                "no_negative_balance",
                when="$event != 'DELETE' AND $after.balance < 0",
                then="THROW 'balance cannot be negative'",
            ),
        ],
    )

    id: str | None = None
    email: str
    balance: int = 0


class AuditLog(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="audit_log", connection="events_demo", extra="allow")

    id: str | None = None


print("events:", [event.name for event in Account.get_events()])
print(Account.get_events()[1])  # repr() leaves out when and then

events: ['email_audit', 'no_negative_balance']
Event(name='no_negative_balance', async_mode=False, retry=None, max_depth=None, comment=None)


`Account.event_ddl()` renders the statements without a connection: names quoted, `then` in a block, the comment an
escaped string literal. `await Account.define_events()` applies them and returns what it applied. It writes
`DEFINE EVENT OVERWRITE`, so it is idempotent: run it at start-up or deployment (`overwrite=False` writes
`IF NOT EXISTS` and keeps an event already defined).

In [3]:
for statement in Account.event_ddl():
    print(statement)

first = await Account.define_events()
second = await Account.define_events()  # idempotent: OVERWRITE
print()
print("applied:", len(first), "statements | the same twice:", first == second == Account.event_ddl())

DEFINE EVENT OVERWRITE `email_audit` ON `account` WHEN $event = 'UPDATE' AND $before.email != $after.email THEN { CREATE audit_log SET account = $value.id, old = $before.email, new = $after.email } COMMENT 'Keeps every e-mail change';
DEFINE EVENT OVERWRITE `no_negative_balance` ON `account` WHEN $event != 'DELETE' AND $after.balance < 0 THEN { THROW 'balance cannot be negative' };

applied: 2 statements | the same twice: True


## 4. Writes fire the events

The events run in the database on every write to `account`. Creating Ada, then changing her balance, writes no audit
row (the event's `WHEN` asks for an e-mail change); changing her e-mail writes one.

In [4]:
ada = Account(id="ada", email="ada@old.example", balance=10)
await ada.save()
await ada.merge(email="ada@new.example")
await ada.merge(balance=20)  # no e-mail change: no audit row

for log in await AuditLog.objects().all():
    row = log.model_extra
    print(f"audit: {row['account']} {row['old']} -> {row['new']}")

audit: account:ada ada@old.example -> ada@new.example


A write that an event refuses raises the SDK's `InternalError` (a `ServerError`), and nothing is written: the stored
balance is unchanged. Deleting Ada fires both events with `$event = 'DELETE'`, which their `WHEN` skip: the delete
passes and writes no audit row.

In [5]:
from surrealdb import ServerError

try:
    await ada.merge(balance=-5)
except ServerError as error:  # the event refused the write
    print(type(error).__name__, "| balance cannot be negative:", "balance cannot be negative" in str(error))

stored = await Account.objects().get("ada")
print("stored balance:", stored.balance)

await stored.delete()
print("accounts:", await Account.objects().all(), "| audit rows:", len(await AuditLog.objects().all()))

InternalError | balance cannot be negative: True
stored balance: 20
accounts: [] | audit rows: 1


## 5. Events without a model

`SurrealDBConnectionManager.define_event(event, table)` applies an `Event` to any table and returns its statement;
`remove_event(event, table)` removes one by name (`REMOVE EVENT IF EXISTS`). Both run on the current connection —
here the `using("events_demo")` block's. The `product` table has no model: a raw query writes it, and the guard refuses
the write all the same. Once the event is removed, the write passes.

In [6]:
stock_guard = Event(
    "stock_guard", when="$event != 'DELETE' AND $after.stock < 0", then="THROW 'stock cannot go below zero'"
)

async with SurrealDBConnectionManager.using("events_demo"):
    print(await SurrealDBConnectionManager.define_event(stock_guard, "product"))
    try:
        await demo.query("CREATE product:widget SET stock = -1;", {})
    except ServerError as error:
        print("refused:", type(error).__name__)

    print(await SurrealDBConnectionManager.remove_event("stock_guard", "product"))
    print("written:", await demo.query("CREATE product:widget SET stock = -1 RETURN VALUE stock;", {}))

DEFINE EVENT OVERWRITE `stock_guard` ON `product` WHEN $event != 'DELETE' AND $after.stock < 0 THEN { THROW 'stock cannot go below zero' };
refused: InternalError
REMOVE EVENT IF EXISTS `stock_guard` ON `product`;
written: [-1]


## 6. Transactions

Every method that applies DDL takes `tx=`. Below, `Account`'s events are removed, then defined again inside a
`transaction()` block, together with writes: the block's update fires `email_audit`, whose row is there once the block
commits. In a buffered transaction (SurrealDB 2.x, HTTP, the embedded engine) the statements are applied at commit, in
order; in an interactive one (WebSocket and SurrealDB 3.x) they are applied in the transaction. A block that raises
leaves nothing defined.

In [7]:
from surreal_orm_lite import SurrealDbError

print(await Account.remove_event("email_audit"))
print(await Account.remove_event("no_negative_balance"))

async with SurrealDBConnectionManager.transaction("events_demo") as tx:
    print("applied in the block:", len(await Account.define_events(tx=tx)), "statements")
    grace = Account(id="grace", email="grace@old.example", balance=5)
    await grace.save(tx=tx)
    await grace.merge(email="grace@new.example", tx=tx)

audit = sorted((log.model_extra["old"], log.model_extra["new"]) for log in await AuditLog.objects().all())
print("audit rows:", audit)

REMOVE EVENT IF EXISTS `email_audit` ON `account`;
REMOVE EVENT IF EXISTS `no_negative_balance` ON `account`;
applied in the block: 2 statements
audit rows: [('ada@old.example', 'ada@new.example'), ('grace@old.example', 'grace@new.example')]


A write the guard refuses inside a `transaction()` block makes the block raise `SurrealDbError`: `transaction()`
cancels it, and nothing of the block is written — neither Linus, saved first, nor Ken.

In [8]:
try:
    async with SurrealDBConnectionManager.transaction("events_demo") as tx:
        await Account(id="linus", email="linus@example.com", balance=5).save(tx=tx)
        await Account(id="ken", email="ken@example.com", balance=-1).save(tx=tx)
except SurrealDbError as error:
    print(type(error).__name__, "| balance cannot be negative:", "balance cannot be negative" in str(error))

print("accounts:", sorted(account.id for account in await Account.objects().all()))

SurrealDbError | balance cannot be negative: True
accounts: ['grace']


## 7. Read the events back

What the ORM defines reads back through v0.30.0's `DatabaseIntrospector`: each event is an `EventState` with its
`when`, `comment` and, since v0.31.0, `async_mode`, `retry` and `max_depth`.

In [9]:
from surreal_orm_lite import DatabaseIntrospector

schema = await DatabaseIntrospector("events_demo", tables=["account"]).introspect()
for name, event in sorted(schema.tables["account"].events.items()):
    print(f"{name}: when={event.when!r} comment={event.comment!r}")
    print(f"    async_mode={event.async_mode} retry={event.retry} max_depth={event.max_depth}")
print("issues:", schema.issues)

email_audit: when="$event = 'UPDATE' AND $before.email != $after.email" comment='Keeps every e-mail change'
    async_mode=False retry=None max_depth=None
no_negative_balance: when="$event != 'DELETE' AND $after.balance < 0" comment=None
    async_mode=False retry=None max_depth=None
issues: ()


`then` is kept as the server renders it, and each line renders a block its own way: SurrealDB 3.x writes a block of
one statement on one line without its `;`, SurrealDB 2.x keeps the `;` (and puts a block of several statements on
several lines). The output below comes from the server the notebook ran on.

In [10]:
for name, event in sorted(schema.tables["account"].events.items()):
    print(f"{name}: THEN {event.then}")

email_audit: THEN { CREATE audit_log SET account = $value.id, old = $before.email, new = $after.email }
no_negative_balance: THEN { THROW 'balance cannot be negative' }


## 8. Asynchronous events (SurrealDB 3.x only)

On SurrealDB 3.x an event with `async_mode=True` (`ASYNC`) runs after the write's transaction commits, so it does not
hold the write up; `retry` (`RETRY`) is how many times SurrealDB retries a failing body, and `max_depth` (`MAXDEPTH`)
how deep a chain of asynchronous events may go. `event.ddl()` renders it on any line. Its row appears a few
milliseconds after the commit, so the cell waits for it (at most 5 s). SurrealDB 2.x and the embedded engine cannot
parse `ASYNC`: there the ORM raises `SurrealDbError` before sending anything, and the cell prints that refusal.

In [11]:
notify = Event(
    "notify_shipping",
    when="$event = 'CREATE'",
    then="CREATE outbox SET shipment = $value.id, at = time::now()",
    async_mode=True,
    retry=3,
)
print(notify.ddl("shipment"))


class Outbox(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="outbox", connection="events_demo", extra="allow")

    id: str | None = None


async def wait_rows(model, n: int, timeout: float = 5.0) -> list:
    """The rows of *model* once there are at least *n*, polled every 20 ms; TimeoutError after *timeout* s."""
    loop = asyncio.get_running_loop()
    deadline = loop.time() + timeout
    while True:
        rows = await model.objects().all()
        if len(rows) >= n:
            return rows
        if loop.time() >= deadline:
            raise TimeoutError(f"{n} row(s) expected within {timeout} s, {len(rows)} read")
        await asyncio.sleep(0.02)


async with SurrealDBConnectionManager.using("events_demo"):
    if major >= 3:
        await SurrealDBConnectionManager.define_event(notify, "shipment")
        await demo.query("CREATE shipment:parcel SET item = 'book';", {})
        rows = await wait_rows(Outbox, 1)
        print("outbox:", [str(row.model_extra["shipment"]) for row in rows])
        state = (await DatabaseIntrospector("events_demo", tables=["shipment"]).introspect()).tables["shipment"]
        read = state.events["notify_shipping"]
        print(f"read back: async_mode={read.async_mode} retry={read.retry} max_depth={read.max_depth}")
    else:
        try:
            await SurrealDBConnectionManager.define_event(notify, "shipment")
        except SurrealDbError as error:  # refused before any DDL
            print(error)

DEFINE EVENT OVERWRITE `notify_shipping` ON `shipment` ASYNC RETRY 3 WHEN $event = 'CREATE' THEN { CREATE outbox SET shipment = $value.id, at = time::now() };
outbox: ['shipment:parcel']
read back: async_mode=True retry=3 max_depth=3


## 9. Stored functions

`define_function()` defines `fn::…` on the current connection and returns the statement; `call_function()` calls it.
`parameters` maps each parameter name, without the `$`, to its SurrealQL kind, in the order of `call_function()`'s
positional `args`; the body is written as a block and `RETURN` gives the result.

In [12]:
async with SurrealDBConnectionManager.using("events_demo"):
    print(
        await SurrealDBConnectionManager.define_function(
            "fn::greet",
            "RETURN string::concat('Hello, ', $name, '!');",
            parameters={"name": "string"},
            returns="string",
            comment="Says hello",
        )
    )
    print(await SurrealDBConnectionManager.call_function("fn::greet", ["Ada"]))

DEFINE FUNCTION OVERWRITE fn::greet($name: string) -> string { RETURN string::concat('Hello, ', $name, '!'); } COMMENT 'Says hello';
Hello, Ada!


Redefining the function (`OVERWRITE`) with two parameters: `call_function(params=…)` passes them by name.
`define_function()` drops the signature `params=` had cached, so the call follows the redefinition at once. Kinds are
written in canonical form (`none | string` would be written `option<string>` too), and a trailing `option<…>`
parameter may be left out of a positional call.

In [13]:
async with SurrealDBConnectionManager.using("events_demo"):
    print(
        await SurrealDBConnectionManager.define_function(
            "fn::greet",
            "RETURN string::concat($greeting ?? 'Hello', ', ', $name, '!');",
            parameters={"name": "string", "greeting": "option<string>"},
            returns="string",
            comment="Says hello, or another greeting",
        )
    )
    print(await SurrealDBConnectionManager.call_function("fn::greet", params={"name": "Ada", "greeting": "Hi"}))
    print(await SurrealDBConnectionManager.call_function("fn::greet", ["Grace"]))  # the option<string> left out

DEFINE FUNCTION OVERWRITE fn::greet($name: string, $greeting: option<string>) -> string { RETURN string::concat($greeting ?? 'Hello', ', ', $name, '!'); } COMMENT 'Says hello, or another greeting';
Hi, Ada!
Hello, Grace!


The function reads back as a `FunctionState` (its parameter names, result kind and comment). `remove_function()`
writes `REMOVE FUNCTION IF EXISTS`; the next call raises `SurrealDbNotFoundError`.

In [14]:
from surreal_orm_lite import SurrealDbNotFoundError

functions = (await DatabaseIntrospector("events_demo", tables=[]).introspect()).functions
greet = functions["fn::greet"]
print("read back:", greet.name, "| parameters", greet.parameters, "| returns", greet.returns)
print("comment:  ", repr(greet.comment))

async with SurrealDBConnectionManager.using("events_demo"):
    print(await SurrealDBConnectionManager.remove_function("fn::greet"))
    try:
        await SurrealDBConnectionManager.call_function("fn::greet", ["Ada"])
    except SurrealDbNotFoundError as error:
        print(type(error).__name__)

read back: fn::greet | parameters ('name', 'greeting') | returns string
comment:   'Says hello, or another greeting'
REMOVE FUNCTION IF EXISTS fn::greet;
SurrealDbNotFoundError


## 10. Cleanup

Remove the `examples_events` database (its events and functions with it), unregister `events_demo` —
`remove_connection()` closes its client — then close every connection with `close_all_connections()`.

In [15]:
await apply_ddl(demo, "REMOVE DATABASE IF EXISTS examples_events;")
await SurrealDBConnectionManager.remove_connection("events_demo")
print("Connections:", SurrealDBConnectionManager.list_connections())
await SurrealDBConnectionManager.close_all_connections()
print("Done")

Connections: ['default']
Done
